# Vision Language Model (VLM) Fine-Tuning with Unsloth
This notebook demonstrates how to prepare custom image-text datasets and fine-tune Vision Language Models (VLMs) like Qwen2-VL, LLaVA, Pixtral, and others using Unsloth for fast and low-VRAM training.


## 1. Environment Setup
Installing required libraries for the datasets and image processing.


In [1]:
!pip install -U datasets huggingface_hub pillow

| Library           | Single-line use                                                                              |
| ----------------- | -------------------------------------------------------------------------------------------- |
| `datasets`        | Used to load, create, preprocess, and manage datasets for training or fine-tuning models.    |
| `huggingface_hub` | Used to download/upload models, tokenizers, datasets, and checkpoints from Hugging Face Hub. |
| `pillow`          | Used to open, process, resize, and manipulate images in Python.                              |


## 2. Imports and Workspace Configuration
Importing datasets tools, PIL for image manipulation, and Hugging Face Hub dependencies.


In [2]:
import os
from datasets import Dataset, Features, Value, Image

| Code        | Single-line description                                                                |
| ----------- | -------------------------------------------------------------------------------------- |
| `import os` | Used to work with file paths, folders, and environment variables.                      |
| `Dataset`   | Used to create a Hugging Face dataset from Python data.                                |
| `Features`  | Used to define the schema/structure of dataset columns.                                |
| `Value`     | Used to define text, number, or string column types in the dataset.                    |
| `Image`     | Used to define an image column in the dataset.                                         |
| `login`     | Used to authenticate with Hugging Face Hub for uploading or accessing models/datasets. |


## 3. Preparing Custom Dataset (Local Images)
### 3.1 Load Local Images
Locating and sorting the image paths in the local workspace directory.


In [3]:
# 2) Your image folder
IMG_DIR = "./iphone_imgs"

In [4]:
image_files = sorted([
    os.path.join(IMG_DIR, f)
    for f in os.listdir(IMG_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
])

In [5]:
image_files

['./iphone_imgs/image01.jpg',
 './iphone_imgs/image02.jpg',
 './iphone_imgs/image03.jpg',
 './iphone_imgs/image04.jpg',
 './iphone_imgs/image05.jpg']

### 3.2 Define Captions and Instructions
Configuring captions matching the images and the VLM system instructions.


In [6]:
# 3) Captions for images
captions = [
    "A white iPhone shown from the back and front on a white background.",
    "A dark blue iPhone shown from the back with a side view on a transparent background.",
    "An orange iPhone shown from the back and front on a white background.",
    "Three iPhones in white, orange, and dark blue shown together from the back.",
    "An orange iPhone shown from the back on a transparent background.",
]

In [7]:
# assert len(image_files) == len(captions), "Images and captions count must match!"

In [8]:
instruction = "Describe this iPhone product image in one sentence."

### 3.3 Construct Dataset Structure
Structuring the row lists containing the image objects and the caption text.


In [9]:
rows = []

In [10]:
for img_path, cap in zip(image_files, captions):
    rows.append({
        "image": img_path,
        "text": cap,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": instruction},
                    {"type": "image", "image": img_path},
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": cap},
                ],
            },
        ],
    })

In [11]:
rows

[{'image': './iphone_imgs/image01.jpg',
  'text': 'A white iPhone shown from the back and front on a white background.',
  'messages': [{'role': 'user',
    'content': [{'type': 'text',
      'text': 'Describe this iPhone product image in one sentence.'},
     {'type': 'image', 'image': './iphone_imgs/image01.jpg'}]},
   {'role': 'assistant',
    'content': [{'type': 'text',
      'text': 'A white iPhone shown from the back and front on a white background.'}]}]},
 {'image': './iphone_imgs/image02.jpg',
  'text': 'A dark blue iPhone shown from the back with a side view on a transparent background.',
  'messages': [{'role': 'user',
    'content': [{'type': 'text',
      'text': 'Describe this iPhone product image in one sentence.'},
     {'type': 'image', 'image': './iphone_imgs/image02.jpg'}]},
   {'role': 'assistant',
    'content': [{'type': 'text',
      'text': 'A dark blue iPhone shown from the back with a side view on a transparent background.'}]}]},
 {'image': './iphone_imgs/imag

### 3.4 Creating Hugging Face Dataset WITH Explicit Features
Mapping column data types explicitly (e.g. converting paths to Image features).<br>
Images will be uploaded as images


In [12]:
features = Features({
    "image": Image(),
    "text": Value("string"),
    "messages": Value("string"),  # keep as JSON string OR store raw python objects separately
})

In [13]:
ds = Dataset.from_list(rows, features=features)
ds

Dataset({
    features: ['image', 'text', 'messages'],
    num_rows: 5
})

| Column                        | Use                                                                             |
| ----------------------------- | ------------------------------------------------------------------------------- |
| `"image": Image()`            | Creates an image column where image files/paths are stored as image data.       |
| `"text": Value("string")`     | Creates a text column for captions, labels, or descriptions.                    |
| `"messages": Value("string")` | Creates a string column to store chat/instruction data |


In [14]:
# If you want to store messages as real objects, easiest is to NOT force Features for messages.
ds = Dataset.from_list(rows)  # messages stored as nested objects
ds

Dataset({
    features: ['image', 'text', 'messages'],
    num_rows: 5
})

### 3.5 Split Dataset and Upload to Hugging Face Hub
Splitting the generated dataset into train/test subsets and pushing them to Hugging Face Hub.


In [15]:
# 4) Make train/test splits (with only 5 images, keep test=1)
splits = ds.train_test_split(test_size=1, seed=3407)

In [16]:
# 5) Push to Hub
REPO_ID = "AreebAhmxd/iphone_vlm_img"

In [17]:
# from huggingface_hub import login
# login()

In [18]:
# splits["train"].push_to_hub(REPO_ID, split="train")

In [19]:
# splits["test"].push_to_hub(REPO_ID, split="test")

In [20]:
# print("Uploaded:", REPO_ID)

### 3.5.1 Creating Hugging Face Dataset WITHOUT Explicit Features
Images will be uploaded as image path strings

In [21]:
from datasets import Dataset
ds = Dataset.from_list(rows)

In [22]:
ds

Dataset({
    features: ['image', 'text', 'messages'],
    num_rows: 5
})

In [23]:
REPO_ID = "AreebAhmxd/iphone_vlm"

In [24]:
# ds.push_to_hub(REPO_ID)

## 4. Working with Public Vision Datasets
Loading and inspecting benchmark datasets like ChartQA from Hugging Face.


In [25]:
from datasets import load_dataset

In [26]:
dataset = load_dataset("HuggingFaceM4/ChartQA")

data/train-00000-of-00003-49492f364babfa(…):   0%|          | 0.00/219M [00:00<?, ?B/s]

data/train-00001-of-00003-7302bae5e425bb(…):   0%|          | 0.00/311M [00:00<?, ?B/s]

data/train-00002-of-00003-194c9400785577(…):   0%|          | 0.00/315M [00:00<?, ?B/s]

data/val-00000-of-00001-0f11003c77497969(…):   0%|          | 0.00/50.2M [00:00<?, ?B/s]

data/test-00000-of-00001-e2cd0b7a0f9eb20(…):   0%|          | 0.00/68.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/28299 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/1920 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2500 [00:00<?, ? examples/s]

In [27]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['image', 'query', 'label', 'human_or_machine'],
        num_rows: 28299
    })
    val: Dataset({
        features: ['image', 'query', 'label', 'human_or_machine'],
        num_rows: 1920
    })
    test: Dataset({
        features: ['image', 'query', 'label', 'human_or_machine'],
        num_rows: 2500
    })
})


In [28]:
train_ds = load_dataset("HuggingFaceM4/ChartQA", split="train")

In [29]:
print(train_ds)

Dataset({
    features: ['image', 'query', 'label', 'human_or_machine'],
    num_rows: 28299
})


In [30]:
print(train_ds[0])

{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=422x359 at 0x7C8705FFCEF0>, 'query': 'Is the value of Favorable 38 in 2015?', 'label': ['Yes'], 'human_or_machine': 0}


## Datasets Used for Vision-Language Model (VLM) Fine-Tuning

| Dataset | Link | Purpose |
|----------|------|---------|
| **LLaVA-Instruct-150K** | https://huggingface.co/datasets/liuhaotian/LLaVA-Instruct-150K | Visual instruction tuning dataset containing image-question-answer conversations. |
| **ChartQA** | https://huggingface.co/datasets/HuggingFaceM4/ChartQA | Question-answering dataset for understanding charts and graphs. |
| **Flickr30k** | https://huggingface.co/datasets/nlphuji/flickr30k | Image-captioning dataset containing images paired with natural language descriptions. |
| **LaTeX OCR** | https://huggingface.co/datasets/unsloth/LaTeX_OCR | OCR dataset for converting images of mathematical expressions into LaTeX. |

---

## LLaVA-Instruct-150K Image Source

The **LLaVA-Instruct-150K** dataset contains only the **annotations and conversations**. The corresponding images are **not included** in the dataset.

The images must be downloaded separately from the **MS COCO 2017 `train2017`** dataset, as specified in the official LLaVA repository. :contentReference[oaicite:0]{index=0}

**LLaVA-Instruct-150K dataset**

https://huggingface.co/datasets/liuhaotian/LLaVA-Instruct-150K

**MS COCO Train 2017 images**

http://images.cocodataset.org/zips/train2017.zip

---

## Meaning of `question_type`

| Value | Meaning |
|-------|---------|
| `0` | Human-generated question |
| `1` | Machine-generated question |

## 5. Fine-Tuning Setup
### 5.1 Define Models and Registry


#### Vision-Language Models Supported by Unsloth

| Model Family | Model |
|--------------|-------|
| **Qwen 2.5 VL** | `unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit` |
| | `unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit` |
| | `unsloth/Qwen2.5-VL-32B-Instruct-bnb-4bit` |
| | `unsloth/Qwen2.5-VL-72B-Instruct-bnb-4bit` |
| **Qwen 2 VL** | `unsloth/Qwen2-VL-2B-Instruct-bnb-4bit` |
| | `unsloth/Qwen2-VL-7B-Instruct-bnb-4bit` |
| | `unsloth/Qwen2-VL-72B-Instruct-bnb-4bit` |
| **Qwen 2.5 Omni (Multimodal)** | `unsloth/Qwen2.5-Omni-3B-Instruct-bnb-4bit` |
| | `unsloth/Qwen2.5-Omni-7B-Instruct-bnb-4bit` |
| **Llama Vision** | `unsloth/Llama-3.2-11B-Vision-Instruct-bnb-4bit` |
| | `unsloth/Llama-3.2-90B-Vision-Instruct-bnb-4bit` |
| **Gemma Vision** | `unsloth/MedGemma-4B-Vision-Instruct-bnb-4bit` |
| | `unsloth/MedGemma-27B-Vision-Instruct-bnb-4bit` |
| **Mistral Vision** | `unsloth/Pixtral-12B-2409-bnb-4bit` |
| **LLaVA** | `unsloth/llava-1.5-7b-hf-bnb-4bit` |
| | `unsloth/llava-v1.6-mistral-7b-hf-bnb-4bit` |

#### Model Identifiers

| Model | Identifier |
|-------|------------|
| **Qwen 2 VL 2B** | `qwen2_vl_2b` |
| **Qwen 2.5 VL 3B** | `qwen25_vl_3b` |
| **Qwen 2.5 Omni 3B** | `qwen25_omni_3b` |
| **MedGemma 4B Vision** | `medgemma_4b_vision` |
| **Qwen 2.5 VL 7B** | `qwen25_vl_7b` |
| **Qwen 2 VL 7B** | `qwen2_vl_7b` |
| **Qwen 2.5 Omni 7B** | `qwen25_omni_7b` |
| **LLaVA 1.5 7B** | `llava15_7b` |
| **LLaVA 1.6 Mistral 7B** | `llava16_mistral_7b` |

### 5.2 Install Packages for Training
Installing training dependencies compatible with Unsloth and modern VLMs.


In [31]:
# --- Install (Colab) ---
!pip -q install "transformers==4.57.1" --upgrade
!pip -q install --no-deps trl==0.22.2
!pip -q install unsloth unsloth_zoo bitsandbytes accelerate peft triton
!pip -q install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 544.8/544.8 kB 14.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

### 5.3 Universal VLM Fine-Tuning Architecture
Setup class structures and imports for training.


#### Universal Unsloth Vision Fine-Tuning Pipeline
#### Features

- Supports multiple Vision-Language Models (VLMs), including:
  - Qwen VL
  - LLaVA
  - Pixtral
  - MedGemma
  - Llama Vision
- Dynamically selects the model at runtime.
- Dynamically selects the training dataset.
- Trains on a subset of **150 samples**.
- Runs for **2 training epochs**.

In [32]:
# Disable hf_transfer to avoid download issues and AttributeError in huggingface_hub
import unsloth
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

import huggingface_hub.constants
if not hasattr(huggingface_hub.constants, "HF_HUB_ENABLE_HF_TRANSFER"):
    huggingface_hub.constants.HF_HUB_ENABLE_HF_TRANSFER = False
import os
import torch
from dataclasses import dataclass
from typing import Dict

from datasets import load_dataset
from transformers import TextStreamer
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


### 5.4 Model Registry
Mapping shorthand model names to official Hugging Face repositories.


In [33]:
# ==========================================================
# Model Registry
# ==========================================================
MODEL_REGISTRY = {
    "qwen2_vl_2b": "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit",
    "qwen25_vl_3b": "unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit",
    "llava15_7b": "unsloth/llava-1.5-7b-hf-bnb-4bit",
    "pixtral_12b": "unsloth/Pixtral-12B-2409-bnb-4bit",
    "medgemma_4b": "unsloth/MedGemma-4B-Vision-Instruct-bnb-4bit",
}

### 5.5 Dataset Registry
Standardizing loading parameters for different vision-ready datasets.


In [34]:
# ==========================================================
# Dataset Registry (Vision-ready datasets)
# ==========================================================
DATASET_REGISTRY = {
    "latex_ocr": {
        "name": "unsloth/LaTeX_OCR",
        "split": "train",
        "image_key": "image",
        "text_key": "text",
        "instruction": "Write the LaTeX representation for this image."
    },
    "flickr30k": {
        "name": "nlphuji/flickr30k",
        "split": "train",
        "image_key": "image",
        "text_key": "caption",
        "instruction": "Describe the image."
    },

    "iphone_custom": {
    "name": "AreebAhmxd/iphone_vlm",
    "split": "train",
    "image_key": "image",
    "text_key": "text",
    "instruction": "Describe this iPhone product image in one sentence."
  },
}

### 5.6 Configuration
Setting hyper-parameters such as epochs, learning rate, and sequence lengths.


In [35]:
# ==========================================================
# Config
# ==========================================================
@dataclass
class VisionFTConfig:
    model_key: str = "qwen2_vl_2b"
    dataset_key: str = "iphone_custom"

    subset_rows: int = 150
    eval_ratio: float = 0.1 #10% data for evaluation
    seed: int = 3407

    # LoRA
    r: int = 16
    lora_alpha: int = 16
    lora_dropout: float = 0.0

    # Training
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 4
    num_train_epochs: int = 2
    learning_rate: float = 2e-4
    logging_steps: int = 10
    weight_decay: float = 0.001
    max_length: int = 2048

    output_dir: str = "outputs"
    save_dir: str = "vlm_lora_output"

| Parameter                              | One-line description                                                    |
| -------------------------------------- | ----------------------------------------------------------------------- |
| `model_key: str = "qwen2_vl_2b"`       | Defines which vision-language model will be used for fine-tuning.       |
| `dataset_key: str = "latex_ocr"`       | Defines which dataset will be used for training.                        |
| `subset_rows: int = 150`               | Uses only 150 rows from the dataset for quick demo/testing.             |
| `eval_ratio: float = 0.1`              | Keeps 10% of the data for evaluation/validation.                        |
| `seed: int = 3407`                     | Fixes randomness so results are reproducible.                           |
| `r: int = 16`                          | LoRA rank; controls how many trainable low-rank parameters are added.   |
| `lora_alpha: int = 16`                 | LoRA scaling factor; controls the strength of LoRA updates.             |
| `lora_dropout: float = 0.0`            | Dropout applied inside LoRA layers to reduce overfitting.               |
| `per_device_train_batch_size: int = 2` | Number of samples processed per GPU/device in one training step.        |
| `gradient_accumulation_steps: int = 4` | Accumulates gradients for 4 steps before updating weights.              |
| `num_train_epochs: int = 2`            | Trains the model for 2 full passes over the training dataset.           |
| `learning_rate: float = 2e-4`          | Controls how fast the model weights are updated during training.        |
| `logging_steps: int = 10`              | Logs training metrics after every 10 steps.                             |
| `weight_decay: float = 0.001`          | Regularization value used to reduce overfitting.                        |
| `max_length: int = 2048`               | Maximum token length allowed for model input/output sequence.           |
| `output_dir: str = "outputs"`          | Folder where training outputs/checkpoints can be saved.                 |
| `save_dir: str = "vlm_lora_output"`    | Final folder where the trained LoRA adapter/model output will be saved. |


In [36]:
# from datasets import load_dataset
# dataset = load_dataset("HuggingFaceM4/ChartQA")
# print(dataset)

In [37]:
# train_ds = load_dataset("HuggingFaceM4/ChartQA", split="train")
# print(train_ds[0])

### 5.7 Universal VLM Trainer Class
Defining the main object that manages loading, dataset formatting, SFT config setup, and validation.


In [38]:
# ==========================================================
# Trainer Class
# ==========================================================
class VisionFineTuner:

    def __init__(self, cfg: VisionFTConfig):
        self.cfg = cfg

        # Get model and dataset details from registry
        self.model_name = MODEL_REGISTRY[cfg.model_key]
        self.dataset_info = DATASET_REGISTRY[cfg.dataset_key]

        self.model = None
        self.tokenizer = None
        self.train_ds = None
        self.eval_ds = None
        self.trainer = None

    # -----------------------------
    # Load Model + Apply LoRA
    # -----------------------------
    def load_model(self):
        print("Loading model:", self.model_name)

        import huggingface_hub.constants
        if not hasattr(huggingface_hub.constants, 'HF_HUB_ENABLE_HF_TRANSFER'):
            huggingface_hub.constants.HF_HUB_ENABLE_HF_TRANSFER = False
        import os
        os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'


        # Download locally to bypass 404 errors
        from huggingface_hub import snapshot_download
        local_dir = snapshot_download(repo_id=self.model_name, ignore_patterns=['additional_chat_templates*'])
        self.model, self.tokenizer = FastVisionModel.from_pretrained(
            local_dir,
            load_in_4bit=True,
            use_gradient_checkpointing="unsloth",
        )

        self.model = FastVisionModel.get_peft_model(
            self.model,
            finetune_vision_layers=True,
            finetune_language_layers=True,
            finetune_attention_modules=True,
            finetune_mlp_modules=True,
            r=self.cfg.r,
            lora_alpha=self.cfg.lora_alpha,
            lora_dropout=self.cfg.lora_dropout,
            bias="none",
            random_state=self.cfg.seed,
        )

        print("Model loaded and LoRA applied.")
        return self

    # -----------------------------
    # Prepare Dataset
    # -----------------------------
    def prepare_data(self):
        print("Loading dataset:", self.dataset_info["name"])

        raw = load_dataset(
            self.dataset_info["name"],
            split=self.dataset_info["split"]
        )

        # Use small subset for demo/testing
        raw = raw.select(range(min(self.cfg.subset_rows, len(raw))))

        instruction = self.dataset_info["instruction"]
        image_key = self.dataset_info["image_key"]
        text_key = self.dataset_info["text_key"]

        def format_sample(example):
            return {
                "messages": [
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": instruction},
                            {"type": "image", "image": example[image_key]},
                        ],
                    },
                    {
                        "role": "assistant",
                        "content": [
                            {"type": "text", "text": str(example[text_key])}
                        ],
                    },
                ]
            }

        ds = raw.map(
            format_sample,
            remove_columns=raw.column_names
        )

        splits = ds.train_test_split(
            test_size=self.cfg.eval_ratio,
            seed=self.cfg.seed
        )

        self.train_ds = splits["train"]
        self.eval_ds = splits["test"]

        print("Train samples:", len(self.train_ds))
        print("Eval samples:", len(self.eval_ds))

        return self

    # -----------------------------
    # Build Trainer
    # -----------------------------
    def build_trainer(self):
        print("Building trainer...")

        FastVisionModel.for_training(self.model)

        training_args = SFTConfig(
            per_device_train_batch_size=self.cfg.per_device_train_batch_size,
            gradient_accumulation_steps=self.cfg.gradient_accumulation_steps,
            num_train_epochs=self.cfg.num_train_epochs,
            learning_rate=self.cfg.learning_rate,
            logging_steps=self.cfg.logging_steps,
            optim="adamw_8bit",
            weight_decay=self.cfg.weight_decay,
            seed=self.cfg.seed,
            output_dir=self.cfg.output_dir,
            report_to="none",

            # Important for vision-language fine-tuning
            remove_unused_columns=False,
            dataset_text_field="",
            dataset_kwargs={"skip_prepare_dataset": True},

            max_length=self.cfg.max_length,
        )

        self.trainer = SFTTrainer(
            model=self.model,
            tokenizer=self.tokenizer,
            data_collator=UnslothVisionDataCollator(
                self.model,
                self.tokenizer
            ),
            train_dataset=self.train_ds,
            eval_dataset=self.eval_ds,
            args=training_args,
        )

        print("Trainer ready.")
        return self

    # -----------------------------
    # Train Model
    # -----------------------------
    def train(self):
        print("Training started for", self.cfg.num_train_epochs, "epochs")
        self.trainer.train()
        print("Training completed.")
        return self

    # -----------------------------
    # Save Model
    # -----------------------------
    def save(self):
        os.makedirs(self.cfg.save_dir, exist_ok=True)

        self.model.save_pretrained(self.cfg.save_dir)
        self.tokenizer.save_pretrained(self.cfg.save_dir)

        print("Model saved to:", self.cfg.save_dir)
        return self

    # -----------------------------
    # Quick Inference Test
    # -----------------------------
    def quick_infer(self, sample_index=0):
        print("Running quick inference...")

        FastVisionModel.for_inference(self.model)

        raw = load_dataset(
            self.dataset_info["name"],
            split=self.dataset_info["split"]
        )

        image = raw[sample_index][self.dataset_info["image_key"]]

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": self.dataset_info["instruction"]},
                    {"type": "image"},
                ],
            }
        ]

        input_text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        inputs = self.tokenizer(
            image,
            input_text,
            add_special_tokens=False,
            return_tensors="pt",
        ).to("cuda")

        streamer = TextStreamer(
            self.tokenizer,
            skip_prompt=True
        )

        self.model.generate(
            **inputs,
            streamer=streamer,
            max_new_tokens=128,
            temperature=1.2,
            do_sample=True,
        )

        return self

    # -----------------------------
    # Full Pipeline Runner
    # -----------------------------
    def run(self):
        self.load_model()
        self.prepare_data()
        self.build_trainer()
        self.train()
        self.save()
        self.quick_infer()

## 6. Execution of Fine-Tuning Pipeline
Initializing the tuner with custom configuration and running the pipeline step-by-step.


In [39]:
cfg = VisionFTConfig(
    model_key="qwen2_vl_2b",
    dataset_key="iphone_custom",
)

In [40]:
# # Create config
# cfg = VisionFTConfig()

In [41]:
# trainer = VisionFineTuner(cfg)
# trainer.run()

In [42]:
# Create fine-tuner object
trainer = VisionFineTuner(cfg)

In [43]:
# Step 1: Load model and tokenizer
trainer.load_model()

Loading model: unsloth/Qwen2-VL-2B-Instruct-bnb-4bit
Error importing huggingface_hub._login: cannot import name '_save_stored_tokens' from 'huggingface_hub.utils._auth' (/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py)
Error importing huggingface_hub._login: cannot import name '_save_stored_tokens' from 'huggingface_hub.utils._auth' (/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py)
==((====))==  Unsloth 2026.6.9: Fast Qwen2_Vl patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth


model.safetensors:   0%|          | 0.00/1.54G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/572 [00:00<?, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


tokenizer_config.json:   0%|          | 0.00/4.33k [00:00<?, ?B/s]

Unsloth: Warning - VLM processor fallback returned None for model_type=qwen2_vl


RuntimeError: Unsloth: The tokenizer is weirdly not loaded? Please check if there is one.

In [ ]:
# Step 2: Prepare training and evaluation dataset
trainer.prepare_data()

In [ ]:
# Step 3: Build SFT trainer
trainer.build_trainer()

In [ ]:
# Step 4: Train model
trainer.train()

In [ ]:
# Step 5: Save trained LoRA model and tokenizer
trainer.save()

In [ ]:
# Step 6: Test model with one sample image
trainer.quick_infer()